### home work of lesson 26 by Vasyl Storchak

In [19]:
import numpy as np
import h5py

from sklearn.metrics import classification_report, confusion_matrix
from keras.layers import Input, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten, Conv2D
from keras.layers import AveragePooling2D, MaxPooling2D, Dropout, GlobalMaxPooling2D, GlobalAveragePooling2D
from keras.models import Model

In [4]:
def load_dataset(train_path, test_path):
    train_dataset = h5py.File(train_path, "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) # your train set features
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) # your train set labels

    test_dataset = h5py.File(test_path, "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) # your test set features
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) # your test set labels

    classes = np.array(test_dataset["list_classes"][:]) # the list of classes

    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes

In [5]:
train_path = 'data/train_happy.h5'
test_path = 'data/test_happy.h5'
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset(train_path, test_path)

# Normalize image vectors
X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

# Reshape
Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print("number of training examples = " + str(X_train.shape[0]))
print("number of test examples = " + str(X_test.shape[0]))
print("X_train shape: " + str(X_train.shape))
print("Y_train shape: " + str(Y_train.shape))
print("X_test shape: " + str(X_test.shape))
print("Y_test shape: " + str(Y_test.shape))

number of training examples = 600
number of test examples = 150
X_train shape: (600, 64, 64, 3)
Y_train shape: (600, 1)
X_test shape: (150, 64, 64, 3)
Y_test shape: (150, 1)


In [6]:
def model(input_shape):
    X_input = Input(input_shape)
    X = ZeroPadding2D((3, 3))(X_input)

    X = Conv2D(32, (7, 7), strides = (1, 1), name = 'conv0')(X)
    X = BatchNormalization(axis = 3, name = 'bn0')(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((2, 2), name='max_pool')(X)

    X = Flatten()(X)
    X = Dense(1, activation='sigmoid', name='fc')(X)

    model = Model(inputs = X_input, outputs = X, name='HappyModel')
    
    return model

In [11]:
def HappyModel(input_shape, activation="relu", dropout_rate=0.5):
    X_input = Input(input_shape)

    X = ZeroPadding2D((3, 3))(X_input)
    X = Conv2D(32, (7, 7), strides=(1, 1), name='conv0')(X)
    X = BatchNormalization(axis=3, name='bn0')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool0')(X)

    X = Conv2D(64, (3, 3), strides=(1, 1), name='conv1')(X)
    X = BatchNormalization(axis=3, name='bn1')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool1')(X)

    X = Conv2D(128, (3, 3), strides=(1, 1), name='conv2')(X)
    X = BatchNormalization(axis=3, name='bn2')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool2')(X)

    X = Flatten()(X)
    X = Dense(128, activation=activation, name='fc1')(X)
    X = Dropout(dropout_rate)(X)

    X = Dense(1, activation='sigmoid', name='fc_out')(X)

    model = Model(inputs=X_input, outputs=X, name='SignClassifier')

    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    
    return model

In [17]:
model = HappyModel(input_shape=(64,64,3), 
                   activation="relu", 
                   dropout_rate=0.5)

history = model.fit(X_train, 
                    Y_train,
                    validation_data=(X_test, Y_test),
                    epochs=200,
                    batch_size=200,
                    verbose=1)

loss, acc = model.evaluate(X_test, 
                           Y_test, 
                           verbose=0)
print(f"Accuracy: {acc:.4f}")

Epoch 1/200


2025-08-17 21:03:12.836490: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 475.38MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-17 21:03:12.836528: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 828.52MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-17 21:03:12.917601: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:549] Omitted potentially buggy algorithm eng14{k25=0} for conv %cudnn-conv-bias-activation.10 = (f32[200,64,30,30]{3,2,1,0}, u8[0]{0}) custom-call(f32[200,32,32,32]{3,2,1,0} %bitcast.7538, f32[64,32,3,3]{3,2,1,0} %bitcast.6405, f32[64]{0} %bitcast.7598), window={si

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4478 - loss: 3.2768

2025-08-17 21:03:17.567631: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 360.53MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-17 21:03:17.567668: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 625.39MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-17 21:03:17.616120: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:549] Omitted potentially buggy algorithm eng14{k25=0} for conv %cudnn-conv-bias-activation.10 = (f32[150,64,30,30]{3,2,1,0}, u8[0]{0}) custom-call(f32[150,32,32,32]{3,2,1,0} %bitcast.590, f32[64,32,3,3]{3,2,1,0} %bitcast.597, f32[64]{0} %bitcast.599), window={size=

3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 616ms/step - accuracy: 0.4529 - loss: 3.4411 - val_accuracy: 0.4400 - val_loss: 0.7011
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4915 - loss: 1.5621 - val_accuracy: 0.4400 - val_loss: 0.7065
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5156 - loss: 0.8530 - val_accuracy: 0.4667 - val_loss: 0.6937
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.5590 - loss: 0.7099 - val_accuracy: 0.5867 - val_loss: 0.6908
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.5758 - loss: 0.6776 - val_accuracy: 0.5933 - val_loss: 0.6839
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.6694 - loss: 0.5972 - val_accuracy: 0.5400 - val_loss: 0.6806
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.7108 - loss: 0.5794 - val_accuracy: 0.6067 - val_loss: 0.6779
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.7306 - loss: 0.5465 - val_accuracy: 0.5467 - val_loss: 0.6754
Epo

In [15]:
model.save("happy.keras")

In [27]:
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9467


In [ ]:
# bad in 20 epoch(upper for 200, for 20 is 0.44), i want make more epoch

In [28]:
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"Accuracy with 200 epoch: {acc:.4f}")

Accuracy with 200 epoch: 0.9467


In [ ]:
# much better

In [43]:
y_pred_probs = model.predict(X_test)
# y_pred = np.argmax(y_pred_probs, axis=1)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [48]:
y_pred_binary = np.where(y_pred_probs > 0.5, 1, 0)

In [49]:
print("\nClassification Report:")
print(classification_report(Y_test, y_pred_binary, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94        66
           1       0.93      0.98      0.95        84

    accuracy                           0.95       150
   macro avg       0.95      0.94      0.95       150
weighted avg       0.95      0.95      0.95       150



In [50]:
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, y_pred_binary))


Confusion Matrix:
[[60  6]
 [ 2 82]]


In [ ]:
# GOOOOOD test acc.

In [40]:
np.set_printoptions(suppress=True)